# Phase 4A — Real final search pipeline (PILOT → FULL)

**Goal:** Actual text query → MiniLM embedding → (optional fitted TruncatedSVD k=100) → normalized BLAS cosine → top-10 real Qwen responses. Compare with the **exact 384-dimensional** baseline. This notebook builds and saves the pipeline; Phase 4B/C will run formal correctness/scaling benchmarks.

**Methodology:** 384d exact and 100d reduced searches are **different retrieval representations**. Reduced-space results are approximate and are NOT required to match exact baseline top-10. Cached corpus embeddings were generated previously with `sentence-transformers/all-MiniLM-L6-v2`. No fabricated output. Use **PILOT=True** to test 1K docs, then **PILOT=False** for 20K; fit SVD only once per selected mode, save artifacts in Drive. CPU for BLAS measurements.

In [ ]:
%pip -q install sentence-transformers scikit-learn joblib threadpoolctl
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, time, joblib
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from threadpoolctl import threadpool_limits

ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
CORPUS=ROOT/'data/raw/qwen_corpus_20k_progress.jsonl'
EMB=ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy'
OUT=ROOT/'phase4_final_system'
MODEL_NAME='sentence-transformers/all-MiniLM-L6-v2'
PILOT=True  # Set False after pilot success; rerun all cells
N=1000 if PILOT else 20000
K=100
TOP_K=10
for d in [OUT,OUT/'models',OUT/'benchmarks',OUT/'figures']:
    d.mkdir(parents=True,exist_ok=True)
assert CORPUS.exists() and EMB.exists(), 'Missing Drive corpus/embedding. Check paths.'
print('Mode:', 'PILOT' if PILOT else 'FULL', '| documents:',N)

## 1. Load and validate real corpus and cached embeddings

Document order MUST be sorted by numeric `id`, matching the original embedding creation. We verify IDs are 0..19999, response text is nonempty, and embedding matrix shape/dtype.

In [ ]:
records=[json.loads(line) for line in CORPUS.open(encoding='utf-8') if line.strip()]
records.sort(key=lambda r:r['id'])
assert len(records)==20000, len(records)
assert [r['id'] for r in records]==list(range(20000)), 'Corpus IDs must be 0..19999.'
assert all(isinstance(r.get('response'),str) and r['response'].strip() for r in records)
E=np.load(EMB,mmap_mode='r')
assert E.shape==(20000,384) and E.dtype==np.float32,(E.shape,E.dtype)
docs=records[:N]
X=normalize(np.asarray(E[:N],dtype=np.float32),axis=1).astype(np.float32)
X=np.ascontiguousarray(X)
assert np.isfinite(X).all()
print('Verified:',len(docs),'real texts, embeddings:',X.shape, 'memory MiB:',round(X.nbytes/2**20,3))

## 2. Fit/load the reduction model and prepare normalized index

SVD is fitted on the **original unnormalized MiniLM vectors** and then its projected outputs are normalized for cosine. The same fitted model must transform incoming query embeddings. Save a separate pilot/full model and reduced index. **Training and loading time are not search latency.**

In [ ]:
tag='pilot_1k' if PILOT else 'full_20k'
model_path=OUT/'models'/f'truncated_svd_k{K}_{tag}.joblib'
index_path=OUT/'models'/f'reduced_index_k{K}_{tag}.npy'
if model_path.exists() and index_path.exists():
    svd=joblib.load(model_path)
    Z=np.load(index_path)
    print('Loaded saved SVD and reduced index')
else:
    print('Fitting TruncatedSVD on real corpus (offline preparation)...')
    t0=time.perf_counter()
    svd=TruncatedSVD(n_components=K,n_iter=7,random_state=42,algorithm='randomized')
    Z=normalize(svd.fit_transform(np.asarray(E[:N],dtype=np.float32)),axis=1).astype(np.float32)
    Z=np.ascontiguousarray(Z)
    joblib.dump(svd,model_path)
    np.save(index_path,Z)
    print('Offline fit+transform seconds:',round(time.perf_counter()-t0,3))
assert Z.shape==(N,K) and np.isfinite(Z).all()
Z=np.ascontiguousarray(Z,dtype=np.float32)
print('Exact index MiB:',round(X.nbytes/2**20,3),'reduced index MiB:',round(Z.nbytes/2**20,3))

## 3. Query embedding → BLAS → top-10 real texts

`search_text` uses the same MiniLM model as the cached corpus. Scores are cosine similarities on L2-normalized vectors. Stable tie-breaker is smaller document ID. The reference 384d index is exact within float32 arithmetic; reduced 100d ranking may differ.

In [ ]:
encoder=SentenceTransformer(MODEL_NAME,device='cpu')
assert encoder.get_sentence_embedding_dimension()==384

def rank_topk(index, q, k=TOP_K):
    q=np.asarray(q,dtype=np.float32).reshape(-1)
    assert index.shape[1]==q.shape[0]
    with threadpool_limits(limits=1):
        scores=index@q
    order=np.lexsort((np.arange(len(scores)),-scores))[:k]
    return [(int(i),float(scores[i])) for i in order]

def search_text(query_text,k=TOP_K):
    assert isinstance(query_text,str) and query_text.strip()
    start=time.perf_counter()
    q384=encoder.encode([query_text],convert_to_numpy=True,normalize_embeddings=False)[0].astype(np.float32)
    embed_ms=(time.perf_counter()-start)*1000
    q_exact=normalize(q384.reshape(1,-1),axis=1)[0]
    q_reduced=normalize(svd.transform(q384.reshape(1,-1)),axis=1)[0].astype(np.float32)
    start=time.perf_counter()
    exact=rank_topk(X,q_exact,k)
    exact_ms=(time.perf_counter()-start)*1000
    start=time.perf_counter()
    reduced=rank_topk(Z,q_reduced,k)
    reduced_ms=(time.perf_counter()-start)*1000
    overlap=len({i for i,_ in exact}&{i for i,_ in reduced})/k
    return {'query':query_text,'embedding_ms':embed_ms,'exact_search_ms':exact_ms,
            'reduced_search_ms':reduced_ms,'topk_overlap':overlap,
            'exact':exact,'reduced':reduced}

query='How can machine learning improve cybersecurity threat detection?'
result=search_text(query)
print('Query:',result['query'])
print('Embedding:',round(result['embedding_ms'],2),'ms | Exact search:',round(result['exact_search_ms'],3),
      'ms | Reduced search:',round(result['reduced_search_ms'],3),'ms')
print('Top-k overlap:',round(result['topk_overlap'],2))
for mode in ['exact','reduced']:
    print('\\n',mode.upper(),'top 10')
    for rank,(i,score) in enumerate(result[mode],1):
        print(f'{rank:2d}. doc_id={i:5d}, cosine={score:.5f} | {docs[i]["response"][:160].replace(chr(10)," ")}')

## 4. Save run metadata (not a formal performance benchmark)

The single interactive query is a functional demo, **not** a five-repeat benchmark. Phase 4B will check numerical correctness and top-k on multiple queries. Phase 4C will benchmark scaling 1K/5K/10K/20K with >=50 queries and >=5 timed repetitions after warmup.

In [ ]:
meta={'phase':'4A','mode':tag,'model':MODEL_NAME,'n_docs':N,'original_dim':384,
      'reduced_dim':K,'top_k':TOP_K,'svd_model_path':str(model_path),
      'reduced_index_path':str(index_path),'demo_query':result['query'],
      'demo_exact_search_ms':result['exact_search_ms'],
      'demo_reduced_search_ms':result['reduced_search_ms'],
      'demo_top10_overlap':result['topk_overlap'],
      'warning':'Demo single query timings are NOT benchmark medians; Phase 4B/C pending.'}
path=OUT/'benchmarks'/f'phase4a_pipeline_metadata_{tag}.json'
path.write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
print('Saved:',path)
print('NEXT: confirm pilot outputs, then set PILOT=False and rerun.')